In [1]:
import os

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\user\AppData\Local\Temp\ipykernel_12796\2431843486.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [3]:
# Step 1: Setup LLM (text generation with HuggingFace)
HF_TOKEN = os.environ.get('HF_TOKEN')
HUGGINGFACE_REPO_ID = 'Qwen/Qwen2.5-72B-Instruct'
huggingface_repo_id = HUGGINGFACE_REPO_ID

def load_llm(huggingface_repo_id): 
    endpoint = HuggingFaceEndpoint(
        repo_id=huggingface_repo_id,
        task='conversational',
        temperature=0.5,
        max_new_tokens=512,
        huggingfacehub_api_token=HF_TOKEN
    )
    llm = ChatHuggingFace(llm=endpoint)
    return llm

In [4]:
# Step 2: Connect LLM with FAISS and Create chain 
CUSTOM_PROMPT_TEMPLATE = """
Use the pieces of information provided in the context to answer user's question.
If you dont know the answer, just say that you dont know, dont try to make up an answer. 
Dont provide anything out of the given context

Context: {context}
Question: {question}

Start the answer directly. No small talk please.
"""

In [5]:
def set_custom_prompt(custom_prompt_template): 
    prompt = PromptTemplate(template=custom_prompt_template, input_variables=['context', 'question'])
    return prompt 

In [6]:
# Load Dataset  
DB_FAISS_PATH = "vectorstore/db_faiss"
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.load_local(DB_FAISS_PATH, embedding_model, allow_dangerous_deserialization=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=load_llm(HUGGINGFACE_REPO_ID),
    chain_type='stuff',
    retriever=db.as_retriever(search_kwargs={'k': 3}),
    return_source_documents=True,
    chain_type_kwargs={'prompt': set_custom_prompt(CUSTOM_PROMPT_TEMPLATE)}
)

In [12]:
user_query = "give me the list of all diseases u know about?? "

response = qa_chain.invoke({
    "query": user_query
})

print("RESULT:", response["result"])
print("SOURCE DOCUMENTS:", response["source_documents"])

RESULT: - Tennis elbow (Epicondylitis)
- Hepatitis A (Epidemic icterus)
- Typhus (Epidemic typhus)
- Rotavirus infections (Epidemic viral gastroenteritis)
- Eosinophilic pneumonia
- Hand-foot-and-mouth disease
- Herpangina
SOURCE DOCUMENTS: [Document(id='b271d78e-8d45-4333-a874-abaac2c10ab9', metadata={'producer': 'GPL Ghostscript 9.10', 'creator': '', 'creationdate': '2017-05-01T10:37:35-07:00', 'moddate': '2017-05-01T10:37:35-07:00', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf', 'total_pages': 759, 'page': 602, 'page_label': '603'}, page_content='Epicondylitis see Tennis elbow\nEpidemic icterus see Hepatitis A\nEpidemic typhus see Typhus\nEpidemic viral gastroenteritis see Rotavirus\ninfections\nGALE ENCYCLOPEDIA OF MEDICINE 21216\nEosinophilic pneumonia'), Document(id='4d66a4e0-c92b-46db-b2e8-c9e9a1855856', metadata={'producer': 'GPL Ghostscript 9.10', 'creator': '', 'creationdate': '2017-05-01T10:37:35-07:0

In [ ]:
user_query = "I have give u encyclopedia then u have only that much less disease?? give me  the more topics u know about?? "

response = qa_chain.invoke({
    "query": user_query
})

print("RESULT:", response["result"])
print("SOURCE DOCUMENTS:", response["source_documents"])

RESULT: The Gale Encyclopedia of Medicine 2 covers nearly 1,700 common medical disorders, conditions, tests, and treatments. This includes high-profile diseases such as AIDS, Alzheimer’s disease, cancer, and heart attack. The encyclopedia provides thorough coverage of each topic in language that laypersons can understand.
SOURCE DOCUMENTS: [Document(id='0d3d2869-d7ea-4d49-882c-de7477d6c20a', metadata={'producer': 'GPL Ghostscript 9.10', 'creator': '', 'creationdate': '2017-05-01T10:37:35-07:00', 'moddate': '2017-05-01T10:37:35-07:00', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf', 'total_pages': 759, 'page': 5, 'page_label': '6'}, page_content='point the reader to related entries in the encyclopedia.\nGALE ENCYCLOPEDIA OF MEDICINE 2 IX\nINTRODUCTION'), Document(id='39b0747e-d1cc-4e4a-b9f9-d5b30a9e53a3', metadata={'producer': 'GPL Ghostscript 9.10', 'creator': '', 'creationdate': '2017-05-01T10:37:35-07:00', 'mod

In [15]:
user_query = "give me the detailed explaantion of Endarterectomy"

response = qa_chain.invoke({
    "query": user_query
})

print("RESULT:", response["result"])
print("SOURCE DOCUMENTS:", response["source_documents"])

RESULT: Endarterectomy is an operation designed to remove or bypass the fatty deposits, or blockage, in an artery that has been narrowed by the buildup of fatty tissue (atherosclerosis). The primary purpose of this procedure is to restore normal blood flow to the part of the body supplied by the affected artery. 

This surgical intervention is commonly performed to treat cerebrovascular disease, where there is a significant reduction of blood supply to the brain (carotid endarterectomy), and peripheral vascular disease, which involves impaired blood supply to the legs. 

For the surgery to be effective, the degree of narrowing in the artery should be at least 70%, but it should not be completely blocked. Prior to the endarterectomy, patients typically undergo angiography, a diagnostic procedure that involves the use of a local anesthetic to visualize the blockage. The actual endarterectomy surgery requires general anesthesia.

After the surgery, patients are required to lie flat in bed